In [1]:
%pip install pyspark        

Note: you may need to restart the kernel to use updated packages.


In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("OlistDataCleaning") \
    .config("spark.driver.host", "127.0.0.1") \
    .getOrCreate()

In [3]:
df = spark.read.csv('olist_order_items_dataset.csv', header=True, inferSchema=True)
df.printSchema()
df.show(5)

root
 |-- order_id: string (nullable = true)
 |-- order_item_id: integer (nullable = true)
 |-- product_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- shipping_limit_date: timestamp (nullable = true)
 |-- price: double (nullable = true)
 |-- freight_value: double (nullable = true)

+--------------------+-------------+--------------------+--------------------+-------------------+-----+-------------+
|            order_id|order_item_id|          product_id|           seller_id|shipping_limit_date|price|freight_value|
+--------------------+-------------+--------------------+--------------------+-------------------+-----+-------------+
|00010242fe8c5a6d1...|            1|4244733e06e7ecb49...|48436dade18ac8b2b...|2017-09-19 09:45:35| 58.9|        13.29|
|00018f77f2f0320c5...|            1|e5f2d52b802189ee6...|dd7ddc04e1b6c2c61...|2017-05-03 11:05:13|239.9|        19.93|
|000229ec398224ef6...|            1|c777355d18b72b67a...|5b51032eddd242adc...|2018-01-18 14:48

In [4]:
from pyspark.sql.types import StringType, StructField, StructType

# Define the schema to read the date column as a string initially
schema = df.schema
date_field_index = df.schema.fieldNames().index('shipping_limit_date')
schema.fields[date_field_index] = StructField('shipping_limit_date', StringType(), True)

# Reread the data with the correct schema
df = spark.read.csv('olist_order_items_dataset.csv', header=True, schema=schema)

# Now, convert to timestamp
from pyspark.sql.functions import to_timestamp
df = df.withColumn('shipping_limit_date', to_timestamp('shipping_limit_date', 'yyyy-MM-dd HH:mm:ss'))
df.printSchema()
df.show(5)

root
 |-- order_id: string (nullable = true)
 |-- order_item_id: integer (nullable = true)
 |-- product_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- shipping_limit_date: timestamp (nullable = true)
 |-- price: double (nullable = true)
 |-- freight_value: double (nullable = true)

+--------------------+-------------+--------------------+--------------------+-------------------+-----+-------------+
|            order_id|order_item_id|          product_id|           seller_id|shipping_limit_date|price|freight_value|
+--------------------+-------------+--------------------+--------------------+-------------------+-----+-------------+
|00010242fe8c5a6d1...|            1|4244733e06e7ecb49...|48436dade18ac8b2b...|2017-09-19 09:45:35| 58.9|        13.29|
|00018f77f2f0320c5...|            1|e5f2d52b802189ee6...|dd7ddc04e1b6c2c61...|2017-05-03 11:05:13|239.9|        19.93|
|000229ec398224ef6...|            1|c777355d18b72b67a...|5b51032eddd242adc...|2018-01-18 14:48

In [5]:
from pyspark.sql.types import FloatType

# Convert price and freight_value to float
df = df.withColumn('price', df['price'].cast(FloatType()))
df = df.withColumn('freight_value', df['freight_value'].cast(FloatType()))
df.printSchema()
df.show(5)

root
 |-- order_id: string (nullable = true)
 |-- order_item_id: integer (nullable = true)
 |-- product_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- shipping_limit_date: timestamp (nullable = true)
 |-- price: float (nullable = true)
 |-- freight_value: float (nullable = true)

+--------------------+-------------+--------------------+--------------------+-------------------+-----+-------------+
|            order_id|order_item_id|          product_id|           seller_id|shipping_limit_date|price|freight_value|
+--------------------+-------------+--------------------+--------------------+-------------------+-----+-------------+
|00010242fe8c5a6d1...|            1|4244733e06e7ecb49...|48436dade18ac8b2b...|2017-09-19 09:45:35| 58.9|        13.29|
|00018f77f2f0320c5...|            1|e5f2d52b802189ee6...|dd7ddc04e1b6c2c61...|2017-05-03 11:05:13|239.9|        19.93|
|000229ec398224ef6...|            1|c777355d18b72b67a...|5b51032eddd242adc...|2018-01-18 14:48:3

In [6]:
df.select("freight_value").summary().show()

+-------+------------------+
|summary|     freight_value|
+-------+------------------+
|  count|            112650|
|   mean|19.990319955615806|
| stddev|15.806405402408497|
|    min|               0.0|
|    25%|             13.08|
|    50%|             16.26|
|    75%|             21.15|
|    max|            409.68|
+-------+------------------+



In [7]:
from pyspark.sql.functions import col

# Remove outliers using IQR method

# Calculate quartiles
quartiles = df.approxQuantile("freight_value", [0.25, 0.75], 0.0)
q1 = quartiles[0]
q3 = quartiles[1]

# Calculate IQR
iqr = q3 - q1

# Define outlier bounds
lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr

# Filter outliers
df_no_outliers = df.filter((col("freight_value") >= lower_bound) & (col("freight_value") <= upper_bound))

print(f"Original count: {df.count()}")
print(f"Count after removing outliers: {df_no_outliers.count()}")

df = df_no_outliers
df.select("freight_value").summary().show()

Original count: 112650
Count after removing outliers: 100516
+-------+------------------+
|summary|     freight_value|
+-------+------------------+
|  count|            100516|
|   mean|16.125684692127997|
| stddev| 5.468564322367002|
|    min|              0.98|
|    25%|             12.76|
|    50%|             15.62|
|    75%|             18.79|
|    max|             33.25|
+-------+------------------+



In [8]:
df.show(5)

+--------------------+-------------+--------------------+--------------------+-------------------+-----+-------------+
|            order_id|order_item_id|          product_id|           seller_id|shipping_limit_date|price|freight_value|
+--------------------+-------------+--------------------+--------------------+-------------------+-----+-------------+
|00010242fe8c5a6d1...|            1|4244733e06e7ecb49...|48436dade18ac8b2b...|2017-09-19 09:45:35| 58.9|        13.29|
|00018f77f2f0320c5...|            1|e5f2d52b802189ee6...|dd7ddc04e1b6c2c61...|2017-05-03 11:05:13|239.9|        19.93|
|000229ec398224ef6...|            1|c777355d18b72b67a...|5b51032eddd242adc...|2018-01-18 14:48:30|199.0|        17.87|
|00024acbcdf0a6daa...|            1|7634da152a4610f15...|9d7a1d34a50524090...|2018-08-15 10:10:18|12.99|        12.79|
|00042b26cf59d7ce6...|            1|ac6c3623068f30de0...|df560393f3a51e745...|2017-02-13 13:57:51|199.9|        18.14|
+--------------------+-------------+------------

In [9]:
# from pyspark.sql.functions import sum, collect_list, first, round

# order_summary = df.groupBy("order_id", "seller_id") \
#     .agg(
#         collect_list("product_id").alias("product_id"),
#         first("shipping_limit_date").alias("shipping_limit_date"),
#         round(sum("price"), 2).alias("total_price"),
#         round(sum("freight_value"), 2).alias("total_freight_value")
#     )

# order_summary.show(10)

In [10]:
# order_summary.printSchema()
# order_summary.show(20, truncate=False)

In [11]:
# Salvar o DataFrame agregado em CSV usando Pandas (evita dependencia do Hadoop no Windows)
import csv
import os
import pandas as pd
from datetime import datetime

order_items_final_df = df.toPandas()
output_dir = os.getcwd()
output_path = os.path.join(output_dir, f"order_items_final_{datetime.now():%Y%m%d_%H%M%S}.csv")
order_items_final_df.to_csv(output_path, index=False, quoting=csv.QUOTE_ALL)
print(f"CSV salvo em: {output_path}")

CSV salvo em: c:\Users\Sofhia\Downloads\archive\order_items_final_20260331_224715.csv
